# Class agreement vs. checkpoint

For each conditional run in `RUN_DIRS` (run output folder -> legend label), read the per-checkpoint `*_g{guidance}.yaml` files that `scripts/class_agreement_sweep.py` writes into `<run dir>/class_agreement/`, and plot how often an ImageNet classifier gives the generated image the same top-1 class as the original val image whose conditioning embedding produced it.

Bars are grouped first by training dataset (one panel per `dataset_type`), then by run label; within a run there is one bar per checkpoint, light to dark from early to late. The gray tick on each bar is the shuffled baseline: agreement between generated and *unrelated* originals, i.e. what agreement looks like by chance given how concentrated the predicted labels are. A bar is only meaningful by how far it sits above its tick.

Runs are given by output folder rather than by config file, since several runs can share one config. To produce the yaml files, run on the server, per run:

```bash
python -m scripts.class_agreement_sweep --run_dir outputs/<run> --classifiers alexnet,resnet50
```

and copy `<run>/class_agreement/*.yaml` back here. For local use -- no GPU, just reading yaml files and plotting.

In [ ]:
import os
import glob

import yaml
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [ ]:
REPO_ROOT = ".."

# Map each run's output folder (relative to REPO_ROOT) to the label its bars are grouped under.
RUN_DIRS = {
    "outputs/diffusion_full_imagenet64_flow_grad_accum_ema_h64_mc192_ch1x2x3x4_T1000_flow_alexnet_fc6_z50": "AlexNet fc6, 50 PCs",
    "outputs/diffusion_full_imagenet64_flow_grad_accum_ema_h64_mc192_ch1x2x3x4_T1000_flow_alexnet_fc6_z200": "AlexNet fc6, 200 PCs",
    "outputs/diffusion_full_imagenet64_flow_grad_accum_ema_h64_mc192_ch1x2x3x4_T1000_flow_dinov2_vitb14_z50": "DINOv2 ViT-B, 50 PCs",
}

CLASSIFIER = "resnet50"         # one of the classifiers the sweep was run with
METRIC = "top1_agreement"       # or "top5_agreement"
GUIDANCE_SCALE = 1.0            # picks the <stem>_g{GUIDANCE_SCALE:g}.yaml files
STEPS = None                    # e.g. [100_000, 200_000] to plot only those checkpoints; None = all

In [ ]:
# One row per (run, checkpoint). A run with no matching yaml (or none scored with CLASSIFIER)
# is warned about and dropped, so the other runs can still be plotted.
rows = []
for run_dir, label in RUN_DIRS.items():
    eval_dir = os.path.join(REPO_ROOT, run_dir, "class_agreement")
    yaml_paths = sorted(glob.glob(os.path.join(eval_dir, f"*_g{GUIDANCE_SCALE:g}.yaml")))
    n_before = len(rows)
    for p in yaml_paths:
        with open(p) as f:
            info = yaml.safe_load(f)
        scores = info["classifiers"].get(CLASSIFIER)
        if scores is None or (STEPS is not None and info["step"] not in STEPS):
            continue
        rows.append({
            "label": label,
            "dataset_type": info["dataset_type"],
            "step": info["step"],
            "value": scores[METRIC],
            "baseline": scores["shuffled_top1_agreement"],
            "n_images": info["n_images"],
        })
    if len(rows) == n_before:
        print(f"[{label}] no *_g{GUIDANCE_SCALE:g}.yaml scored with {CLASSIFIER!r} in {eval_dir} -- skipping")

assert rows, "Nothing to plot -- run scripts/class_agreement_sweep.py for at least one run first."

for r in sorted(rows, key=lambda r: (r["dataset_type"], list(RUN_DIRS.values()).index(r["label"]), r["step"])):
    print(f"{r['dataset_type']:>10} | {r['label']:<28} step {r['step']:>8,}   "
          f"{METRIC} {100 * r['value']:5.1f}%   shuffled {100 * r['baseline']:5.1f}%   (n={r['n_images']})")

In [ ]:
# Chart tokens: sequential blue ramp (early -> late checkpoint), neutral inks.
SURFACE, INK, INK_2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
STEP_CMAP = LinearSegmentedColormap.from_list("steps", ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
BAR_W, GROUP_GAP = 0.8, 1.0

datasets = sorted({r["dataset_type"] for r in rows})
labels_in_order = list(RUN_DIRS.values())
n_bars = [sum(r["dataset_type"] == d for r in rows) for d in datasets]

fig, axes = plt.subplots(
    1, len(datasets), sharey=True, squeeze=False, facecolor=SURFACE,
    figsize=(max(5, 0.42 * sum(n_bars) + 1.8 * len(datasets)), 4.2),
    gridspec_kw={"width_ratios": [max(n, 3) for n in n_bars]},
)

for ax, dataset in zip(axes[0], datasets):
    ax.set_facecolor(SURFACE)
    x, tick_pos, tick_labels = 0.0, [], []
    for label in labels_in_order:
        run_rows = sorted((r for r in rows if r["dataset_type"] == dataset and r["label"] == label),
                          key=lambda r: r["step"])
        if not run_rows:
            continue
        group_start = x
        shades = STEP_CMAP(np.linspace(0, 1, len(run_rows))) if len(run_rows) > 1 else STEP_CMAP([1.0])
        for r, shade in zip(run_rows, shades):
            ax.bar(x, 100 * r["value"], width=BAR_W, color=shade, edgecolor=SURFACE, linewidth=2, zorder=2)
            ax.hlines(100 * r["baseline"], x - BAR_W / 2, x + BAR_W / 2, color=INK_2, linewidth=2, zorder=3)
            tick_pos.append(x)
            tick_labels.append(f"{r['step'] / 1000:g}k")
            x += 1
        # selective direct label: the latest checkpoint's value only
        last = run_rows[-1]
        ax.text(x - 1, 100 * last["value"] + 1.5, f"{100 * last['value']:.0f}%",
                ha="center", va="bottom", fontsize=8, color=INK)
        ax.text((group_start + x - 1) / 2, -0.2, label, transform=ax.get_xaxis_transform(),
                ha="center", va="top", fontsize=9, color=INK)
        x += GROUP_GAP

    ax.set_xticks(tick_pos, tick_labels, rotation=90, fontsize=7, color=MUTED)
    ax.set_title(dataset, fontsize=10, color=INK)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    ax.tick_params(axis="y", colors=MUTED, labelsize=8)
    ax.tick_params(axis="x", length=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)

axes[0][0].set_ylabel(f"{CLASSIFIER} {METRIC.replace('_', ' ')} (%)", color=INK_2)
axes[0][0].set_ylim(0, 100)
fig.legend(
    handles=[Patch(color=STEP_CMAP(0.0), label="early checkpoint"), Patch(color=STEP_CMAP(1.0), label="late checkpoint"),
             Line2D([], [], color=INK_2, linewidth=2, label="shuffled baseline")],
    loc="upper right", ncol=3, frameon=False, fontsize=8, labelcolor=INK_2,
)
fig.suptitle(f"Class agreement, original vs generated (guidance {GUIDANCE_SCALE:g})", x=0.01, ha="left",
             fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0.04, 1, 0.94))
plt.show()